# JODI Oil World Database — Exploration Notebook

This notebook is the canonical entry-point for analysing the consolidated JODI database
produced by [`scripts/update_jodi.py`](../scripts/update_jodi.py).

**What's in the database**

- `jodi_secondary.parquet` — refined products (LPG, gasoline, diesel, jet, fuel oil…),
  one row per `(country, month, product, flow, unit)`. Roughly 15 M rows from 2002 onward.
- `jodi_primary.parquet` — crude oil & NGLs, same shape. Roughly 7 M rows.

**How it gets refreshed**

JODI publishes year-to-date annual CSVs that get refreshed monthly (around the 25th).
A monthly cron / Task Scheduler job runs `python scripts/update_jodi.py` which downloads
the current-year file from JODI, upserts into the existing parquet, and rewrites the
SQLite mirror.

**Sections**

1. Setup & data load
2. JODI code reference (product / flow / unit dictionaries from the 2017 PDF)
3. Reusable query helper
4. Country-level examples
5. Global aggregates — demand, inventories, days-of-cover, seasonality, trade flows, **monthly YoY fingerprint**
6. SQL access example
7. Snowflake hand-off

> **Caveat on global numbers**: JODI covers ~75-80% of world oil demand by reporting
> coverage. Aggregates here are a useful proxy but not a substitute for the IEA's
> full-coverage monthly Oil Market Report.

## 1. Setup & data load

In [1]:
"""Imports + path resolution.

The notebook is designed to be runnable from either the repo root or the notebooks/
folder. We anchor everything to the project root via the location of this notebook.
"""

import sys
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Resolve project root robustly. Jupyter doesn't set __file__, so we detect
# the country_oil_scraper/ directory by walking up from cwd.
def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_jodi.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_jodi.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(
        "Could not locate country_oil_scraper project root from cwd: " + str(here)
    )

PROJECT_ROOT = _resolve_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "jodi"
PARQUET_SECONDARY = PROCESSED_DIR / "jodi_secondary.parquet"
PARQUET_PRIMARY = PROCESSED_DIR / "jodi_primary.parquet"
SQLITE_SECONDARY = PROCESSED_DIR / "jodi_secondary.db"

print(f"Project root : {PROJECT_ROOT}")
print(f"Secondary    : {PARQUET_SECONDARY}  exists={PARQUET_SECONDARY.exists()}")
print(f"Primary      : {PARQUET_PRIMARY}  exists={PARQUET_PRIMARY.exists()}")

Project root : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper
Secondary    : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed\jodi\jodi_secondary.parquet  exists=True
Primary      : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed\jodi\jodi_primary.parquet  exists=True


In [2]:
"""Load both parquets into memory.

Categorical columns keep RAM well under 1 GB even for the 15M-row secondary table.
First load takes ~2-3 seconds; everything downstream is pandas slicing.
"""

df_sec = pd.read_parquet(PARQUET_SECONDARY)
df_pri = pd.read_parquet(PARQUET_PRIMARY)

def _summary(name: str, df: pd.DataFrame) -> dict:
    return {
        "dataset": name,
        "rows": f"{len(df):,}",
        "date_min": df["date"].min().date(),
        "date_max": df["date"].max().date(),
        "n_countries": df["ref_area"].nunique(),
        "n_products": df["energy_product"].nunique(),
        "n_flows": df["flow_breakdown"].nunique(),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
    }

pd.DataFrame([_summary("secondary", df_sec), _summary("primary", df_pri)])

,dataset,rows,date_min,date_max,n_countries,n_products,n_flows,memory_mb
0,secondary,"15,400,334",2002-01-01,2026-03-01,118,10,10,646.3
1,primary,"6,841,200",2002-01-01,2026-03-01,118,4,10,587.2


## 2. JODI code reference

JODI uses short codes (e.g. `GASDIES`, `TOTDEMO`, `KBD`) for products, flows, and units.
The mappings below are transcribed from
[`reference/jodi-oil-wdb-item-names-ver2017.pdf`](../reference/jodi-oil-wdb-item-names-ver2017.pdf).

### Products (`ENERGY_PRODUCT`)

| Primary code | Meaning           | Secondary code | Meaning                        |
|--------------|-------------------|----------------|--------------------------------|
| CRUDEOIL     | Crude oil         | LPG            | Liquefied petroleum gases      |
| NGL          | Natural gas liquids | NAPHTHA      | Naphtha                        |
| OTHERCRUDE   | Other crude       | GASOLINE       | Motor & aviation gasoline      |
| TOTCRUDE     | Total crude       | KEROSENE       | Kerosenes                      |
|              |                   | JETKERO        | of which: kerosene-type jet    |
|              |                   | GASDIES        | Gas/diesel oil                 |
|              |                   | RESFUEL        | Fuel oil                       |
|              |                   | ONONSPEC       | Other oil products             |
|              |                   | TOTPRODS       | Total oil products             |

### Flows (`FLOW_BREAKDOWN`)

| Primary code | Meaning              | Secondary code | Meaning                  |
|--------------|----------------------|----------------|--------------------------|
| INDPROD      | Production           | REFGROUT       | Refinery output          |
| OSOURCES     | From other sources   | RECEIPTS       | Receipts                 |
| TOTIMPSB     | Imports              | TOTIMPSB       | Imports                  |
| TOTEXPSB     | Exports              | TOTEXPSB       | Exports                  |
| TRANSBAK     | Products transferred / Backflows | PTRANSF | Products transferred  |
| DIRECUSE     | Direct use           | IPTRANSF       | Interproduct transfers   |
| STOCKCH      | Stock change         | STOCKCH        | Stock change             |
| STATDIFF     | Statistical diff.    | STATDIFF       | Statistical diff.        |
| REFINOBS     | Refinery intake      | TOTDEMO        | Demand                   |
| CLOSTLV      | Closing stocks       | CLOSTLV        | Closing stocks           |

### Units (`UNIT_MEASURE`)

| Code     | Meaning                                |
|----------|----------------------------------------|
| KBD      | Thousand barrels per day               |
| KBBL     | Thousand barrels                       |
| KL       | Thousand kilolitres                    |
| KTONS    | Thousand metric tons                   |
| CONVBBL  | Conversion factor (barrels / KTONS)    |

### Assessment codes (`ASSESSMENT_CODE`)

| Code | Meaning                                          |
|------|--------------------------------------------------|
| 1    | Reasonable comparability                         |
| 2    | Use with caution / consult metadata              |
| 3    | Not assessed                                     |
| 4    | Under verification                               |

In [3]:
"""Code -> human-readable lookup dicts.

Use these to render friendly labels in charts and tables.
Source: reference/jodi-oil-wdb-item-names-ver2017.pdf
"""

PRODUCTS_PRIMARY = {
    "CRUDEOIL":   "Crude oil",
    "NGL":        "Natural gas liquids",
    "OTHERCRUDE": "Other crude",
    "TOTCRUDE":   "Total crude",
}

PRODUCTS_SECONDARY = {
    "LPG":      "LPG",
    "NAPHTHA":  "Naphtha",
    "GASOLINE": "Motor & aviation gasoline",
    "KEROSENE": "Kerosenes",
    "JETKERO":  "Jet kerosene (subset of kerosenes)",
    "GASDIES":  "Gas/diesel oil",
    "RESFUEL":  "Residual fuel oil",
    "ONONSPEC": "Other oil products",
    "TOTPRODS": "Total oil products",
}

FLOWS_PRIMARY = {
    "INDPROD":  "Production",
    "OSOURCES": "From other sources",
    "TOTIMPSB": "Imports",
    "TOTEXPSB": "Exports",
    "TRANSBAK": "Products transferred / backflows",
    "DIRECUSE": "Direct use",
    "STOCKCH":  "Stock change",
    "STATDIFF": "Statistical difference",
    "REFINOBS": "Refinery intake",
    "CLOSTLV":  "Closing stocks (level)",
}

FLOWS_SECONDARY = {
    "REFGROUT": "Refinery output",
    "RECEIPTS": "Receipts",
    "TOTIMPSB": "Imports",
    "TOTEXPSB": "Exports",
    "PTRANSF":  "Products transferred",
    "IPTRANSF": "Interproduct transfers",
    "STOCKCH":  "Stock change",
    "STATDIFF": "Statistical difference",
    "TOTDEMO":  "Demand",
    "CLOSTLV":  "Closing stocks (level)",
}

UNITS = {
    "KBD":     "Thousand barrels per day",
    "KBBL":    "Thousand barrels",
    "KL":      "Thousand kilolitres",
    "KTONS":   "Thousand metric tons",
    "CONVBBL": "Conversion factor (barrels per KTON)",
}

ASSESSMENT_MEANING = {
    1: "Reasonable comparability",
    2: "Use with caution / consult metadata",
    3: "Not assessed",
    4: "Under verification",
}

print(f"Products  : {len(PRODUCTS_PRIMARY)} primary, {len(PRODUCTS_SECONDARY)} secondary")
print(f"Flows     : {len(FLOWS_PRIMARY)} primary, {len(FLOWS_SECONDARY)} secondary")
print(f"Units     : {len(UNITS)}")
print(f"Quality   : {len(ASSESSMENT_MEANING)} codes")

Products  : 4 primary, 9 secondary
Flows     : 10 primary, 10 secondary
Units     : 5
Quality   : 4 codes


## 3. Reusable query helper

`query_jodi()` removes the boilerplate from every downstream cell. It accepts scalars or
lists for any filter, defaults to `KBD` (the most useful flow unit), and by default
keeps only rows where `value_status == 'valid'` so JODI's missing-data sentinels are
silently dropped.

In [4]:
"""Ergonomic filter for either parquet."""

from typing import Iterable, Optional, Union

ScalarOrList = Optional[Union[str, Iterable[str]]]


def _as_list(x: ScalarOrList) -> Optional[list[str]]:
    if x is None:
        return None
    if isinstance(x, str):
        return [x]
    return list(x)


def query_jodi(
    df: pd.DataFrame,
    ref_area: ScalarOrList = None,
    product:  ScalarOrList = None,
    flow:     ScalarOrList = None,
    unit:     ScalarOrList = "KBD",
    valid_only: bool = True,
    date_min: Optional[str] = None,
    date_max: Optional[str] = None,
) -> pd.DataFrame:
    """Filter a JODI dataframe by any subset of dimensions.

    Examples
    --------
    >>> query_jodi(df_sec, ref_area="IN", product="GASDIES", flow="TOTDEMO")
    >>> query_jodi(df_sec, ref_area=["US", "CN"], product="GASOLINE", flow="TOTDEMO")
    """
    mask = pd.Series(True, index=df.index)

    for col, vals in (
        ("ref_area",       _as_list(ref_area)),
        ("energy_product", _as_list(product)),
        ("flow_breakdown", _as_list(flow)),
        ("unit_measure",   _as_list(unit)),
    ):
        if vals is not None:
            mask &= df[col].isin(vals)

    if valid_only:
        mask &= df["value_status"] == "valid"
    if date_min is not None:
        mask &= df["date"] >= pd.Timestamp(date_min)
    if date_max is not None:
        mask &= df["date"] <= pd.Timestamp(date_max)

    return df.loc[mask].copy()


# Smoke test: India diesel demand should produce ~290 monthly rows back to 2002
_smoke = query_jodi(df_sec, ref_area="IN", product="GASDIES", flow="TOTDEMO")
print(f"Smoke test: {len(_smoke)} rows for IN x GASDIES x TOTDEMO x KBD")
print(f"Date range: {_smoke['date'].min().date()} -> {_smoke['date'].max().date()}")
_smoke.head(3)

Smoke test: 291 rows for IN x GASDIES x TOTDEMO x KBD
Date range: 2002-01-01 -> 2026-03-01


,date,year,month,ref_area,country_name,energy_product,product_canonical,category,flow_breakdown,metric_type,unit_measure,obs_value,value_status,assessment_code,assessment_label,dataset,source_file,updated_at
24337,2002-01-01,2002,1,IN,India,GASDIES,Diesel,Distillates,TOTDEMO,TOTDEMO,KBD,761.6783,valid,3,white,secondary,2002.csv,2026-05-12 14:25:12.640249
77437,2002-02-01,2002,2,IN,India,GASDIES,Diesel,Distillates,TOTDEMO,TOTDEMO,KBD,763.3796,valid,3,white,secondary,2002.csv,2026-05-12 14:25:12.640249
130537,2002-03-01,2002,3,IN,India,GASDIES,Diesel,Distillates,TOTDEMO,TOTDEMO,KBD,819.1771,valid,3,white,secondary,2002.csv,2026-05-12 14:25:12.640249


## 4. Country-level examples

Four illustrative slices of the data. Edit the country / product codes in any cell to
ask your own question.

In [5]:
"""Example 1 - India diesel total demand 2010 onward."""

ts = query_jodi(df_sec, ref_area="IN", product="GASDIES", flow="TOTDEMO",
                date_min="2010-01-01")

fig = px.line(
    ts, x="date", y="obs_value",
    title="India - Diesel total demand (TOTDEMO, KBD)",
    labels={"obs_value": "kb/d", "date": ""},
)
fig.update_layout(height=400, hovermode="x unified")
fig.show()

In [6]:
"""Example 2 - top 10 gasoline demanders for the latest full year."""

# Latest full year = the most recent calendar year for which all 12 months are present.
yearly_complete = (
    df_sec.loc[df_sec["flow_breakdown"] == "TOTDEMO"]
          .groupby("year")["month"]
          .nunique()
)
latest_full_year = int(yearly_complete[yearly_complete == 12].index.max())
print(f"Latest full year of TOTDEMO data: {latest_full_year}")

snap = query_jodi(
    df_sec, product="GASOLINE", flow="TOTDEMO",
    date_min=f"{latest_full_year}-01-01", date_max=f"{latest_full_year}-12-01",
)
top10 = (
    snap.groupby(["ref_area", "country_name"], observed=True)["obs_value"].mean()
        .reset_index()
        .sort_values("obs_value", ascending=False)
        .head(10)
)

fig = px.bar(
    top10, x="country_name", y="obs_value",
    title=f"Top 10 gasoline demanders ({latest_full_year}, mean kb/d)",
    labels={"obs_value": "kb/d", "country_name": ""},
)
fig.update_layout(height=400, xaxis_tickangle=-30)
fig.show()

Latest full year of TOTDEMO data: 2025


In [7]:
"""Example 3 - product mix for one country (stacked area)."""

COUNTRY = "US"  # change me

mix = query_jodi(
    df_sec,
    ref_area=COUNTRY,
    product=[c for c in PRODUCTS_SECONDARY if c != "TOTPRODS"],  # exclude the total
    flow="TOTDEMO",
    date_min="2015-01-01",
)

# Friendly product labels for the legend
mix["product_label"] = mix["energy_product"].map(PRODUCTS_SECONDARY)

fig = px.area(
    mix.sort_values("date"),
    x="date", y="obs_value", color="product_label",
    title=f"{COUNTRY} - product demand mix (kb/d, 2015 onward)",
    labels={"obs_value": "kb/d", "date": "", "product_label": "Product"},
)
fig.update_layout(height=500, hovermode="x unified")
fig.show()

In [8]:
"""Example 4 - sanity check: crude refinery intake (primary) vs total products
refinery output (secondary) for one country. They should track each other closely
because every barrel of crude refined ends up as some product."""

COUNTRY = "US"

intake = (
    query_jodi(df_pri, ref_area=COUNTRY, product="CRUDEOIL", flow="REFINOBS")
    .rename(columns={"obs_value": "Crude refinery intake (REFINOBS)"})
    [["date", "Crude refinery intake (REFINOBS)"]]
)
output = (
    query_jodi(df_sec, ref_area=COUNTRY, product="TOTPRODS", flow="REFGROUT")
    .rename(columns={"obs_value": "Total products refinery output (REFGROUT)"})
    [["date", "Total products refinery output (REFGROUT)"]]
)
joined = intake.merge(output, on="date", how="outer").sort_values("date")

fig = px.line(
    joined.melt(id_vars="date", var_name="series", value_name="kb/d"),
    x="date", y="kb/d", color="series",
    title=f"{COUNTRY} - crude intake vs total products output (kb/d)",
)
fig.update_layout(height=400, hovermode="x unified", legend=dict(orientation="h", y=-0.2))
fig.show()

## 5. Global aggregates

> Coverage caveat: not every JODI reporter submits every month. Each chart shows
> two universes:
> - **All reporters** — sum across all countries that reported a value that month.
> - **Stable panel** — restricted to countries that reported in at least 90% of months
>   over the last 5 years. Smaller absolute volume, smoother trend.
>
> A reporting-coverage line is included so you can see how many countries are in each sum.

### 5.1 Aggregation helper & reporter panels

In [9]:
"""Build the 'stable panel' once for each parquet, then aggregate.

A country is in the stable panel for (df, product, flow, unit) if it reported a valid
value in at least 90% of the months in the lookback window. The lookback defaults to
the last 5 complete years so we don't penalise countries that joined JODI later.
"""

STABLE_PANEL_THRESHOLD = 0.90
STABLE_LOOKBACK_YEARS = 5


def _stable_panel(
    df: pd.DataFrame, product: str, flow: str, unit: str,
    lookback_years: int = STABLE_LOOKBACK_YEARS,
    threshold: float = STABLE_PANEL_THRESHOLD,
) -> set[str]:
    """Return the set of ref_area codes that reported >= threshold of months."""
    end_year = int(df["year"].max())
    start_year = end_year - lookback_years
    window = query_jodi(
        df, product=product, flow=flow, unit=unit,
        date_min=f"{start_year}-01-01", date_max=f"{end_year}-12-01",
    )
    months_total = window["date"].nunique()
    if months_total == 0:
        return set()
    months_per_country = window.groupby("ref_area", observed=True)["date"].nunique()
    qualifying = months_per_country[months_per_country >= threshold * months_total]
    return set(qualifying.index.astype(str))


def aggregate_global(
    df: pd.DataFrame, product: str, flow: str, unit: str = "KBD",
    panel: str = "all",
) -> pd.DataFrame:
    """Sum a (product, flow, unit) slice across countries by month.

    Returns columns: date, obs_value, n_countries.
    """
    sub = query_jodi(df, product=product, flow=flow, unit=unit)
    if panel == "stable":
        members = _stable_panel(df, product, flow, unit)
        sub = sub[sub["ref_area"].astype(str).isin(members)]
    elif panel != "all":
        raise ValueError("panel must be 'all' or 'stable'")
    grouped = sub.groupby("date", as_index=False).agg(
        obs_value=("obs_value", "sum"),
        n_countries=("ref_area", "nunique"),
    )
    return grouped.sort_values("date").reset_index(drop=True)


# Sanity check: print the stable-panel size for total products demand
_panel_demo = _stable_panel(df_sec, "TOTPRODS", "TOTDEMO", "KBD")
print(f"Stable panel for TOTPRODS x TOTDEMO x KBD: {len(_panel_demo)} countries")
print(f"  sample members: {sorted(_panel_demo)[:10]}")

Stable panel for TOTPRODS x TOTDEMO x KBD: 57 countries
  sample members: ['AT', 'AU', 'AZ', 'BE', 'BG', 'BH', 'BN', 'CA', 'CH', 'CL']


### 5.2 Global oil consumption

In [10]:
"""Sum total products demand across all reporters and the stable panel."""

dem_all    = aggregate_global(df_sec, "TOTPRODS", "TOTDEMO", "KBD", panel="all")
dem_stable = aggregate_global(df_sec, "TOTPRODS", "TOTDEMO", "KBD", panel="stable")

# Use a secondary axis for the country-count line so it doesn't squash demand on the chart.
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=dem_all["date"],    y=dem_all["obs_value"]    / 1000, name="All reporters (mb/d)"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=dem_stable["date"], y=dem_stable["obs_value"] / 1000, name="Stable panel (mb/d)"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=dem_all["date"], y=dem_all["n_countries"], name="# reporting countries",
               line=dict(dash="dot"), opacity=0.4),
    secondary_y=True,
)
fig.update_layout(
    title="Global oil products demand (TOTPRODS x TOTDEMO)",
    height=450, hovermode="x unified",
)
fig.update_yaxes(title_text="Demand (million bbl/day)", secondary_y=False)
fig.update_yaxes(title_text="# countries reporting", secondary_y=True, showgrid=False)
fig.show()

# Quick text summary
last = dem_all.iloc[-1]
print(f"Latest month in DB: {last['date'].date()}")
print(f"  All-reporters demand : {last['obs_value']/1000:.2f} mb/d  ({int(last['n_countries'])} countries)")

Latest month in DB: 2026-03-01
  All-reporters demand : 75.98 mb/d  (47 countries)


### 5.3 Global inventories (closing stock levels)

In [11]:
"""Crude (primary) and product (secondary) closing stock levels in million barrels.

JODI reports commercial stocks; strategic petroleum reserves are excluded.
"""

crude_all     = aggregate_global(df_pri, "CRUDEOIL", "CLOSTLV", "KBBL", panel="all")
crude_stable  = aggregate_global(df_pri, "CRUDEOIL", "CLOSTLV", "KBBL", panel="stable")
prods_all     = aggregate_global(df_sec, "TOTPRODS", "CLOSTLV", "KBBL", panel="all")
prods_stable  = aggregate_global(df_sec, "TOTPRODS", "CLOSTLV", "KBBL", panel="stable")

fig = go.Figure()
for label, df, dash in [
    ("Crude - all reporters",     crude_all,    "solid"),
    ("Crude - stable panel",      crude_stable, "dot"),
    ("Products - all reporters",  prods_all,    "solid"),
    ("Products - stable panel",   prods_stable, "dot"),
]:
    fig.add_trace(go.Scatter(
        x=df["date"], y=df["obs_value"] / 1000,
        name=label, mode="lines",
        line=dict(dash=dash),
    ))
fig.update_layout(
    title="Global closing stocks (million barrels) - commercial only",
    yaxis_title="Million bbl",
    height=450, hovermode="x unified",
)
fig.show()

### 5.4 Days of forward demand cover

Two views side-by-side:
- **Crude cover** = crude closing stocks (kbbl) / refinery intake (kb/d). How long
  refineries could keep running at current rates if all crude flow stopped.
- **Product cover** = product closing stocks (kbbl) / total product demand (kb/d).
  How long the world could keep consuming if all refinery output stopped.

In [12]:
"""Days-of-cover for crude (vs intake) and products (vs demand)."""

# Crude side: stocks / refinery intake
crude_stk_all = aggregate_global(df_pri, "CRUDEOIL", "CLOSTLV",  "KBBL", panel="all")
crude_intake_all = aggregate_global(df_pri, "CRUDEOIL", "REFINOBS", "KBD",  panel="all")
crude_stk_st  = aggregate_global(df_pri, "CRUDEOIL", "CLOSTLV",  "KBBL", panel="stable")
crude_intake_st  = aggregate_global(df_pri, "CRUDEOIL", "REFINOBS", "KBD",  panel="stable")

# Product side: stocks / demand
prod_stk_all = aggregate_global(df_sec, "TOTPRODS", "CLOSTLV", "KBBL", panel="all")
prod_dem_all = aggregate_global(df_sec, "TOTPRODS", "TOTDEMO", "KBD",  panel="all")
prod_stk_st  = aggregate_global(df_sec, "TOTPRODS", "CLOSTLV", "KBBL", panel="stable")
prod_dem_st  = aggregate_global(df_sec, "TOTPRODS", "TOTDEMO", "KBD",  panel="stable")


def _days_of_cover(stocks: pd.DataFrame, flow: pd.DataFrame, label: str) -> pd.DataFrame:
    """Days = stock_kbbl / flow_kbd. Aligns on date; flow=0 -> NaN to avoid div-by-zero."""
    s = stocks.set_index("date")["obs_value"]
    f = flow.set_index("date")["obs_value"].replace(0, np.nan)
    return pd.DataFrame({"date": s.index, "days": (s / f).reindex(s.index).values, "series": label})


crude_all_df    = _days_of_cover(crude_stk_all, crude_intake_all, "All reporters")
crude_stable_df = _days_of_cover(crude_stk_st,  crude_intake_st,  "Stable panel")
prod_all_df     = _days_of_cover(prod_stk_all,  prod_dem_all,     "All reporters")
prod_stable_df  = _days_of_cover(prod_stk_st,   prod_dem_st,      "Stable panel")

fig = make_subplots(rows=1, cols=2, shared_yaxes=False,
                    subplot_titles=("Crude: stocks / refinery intake",
                                    "Products: stocks / demand"))
for df, name, dash in [(crude_all_df, "All reporters", "solid"),
                       (crude_stable_df, "Stable panel", "dot")]:
    fig.add_trace(go.Scatter(x=df["date"], y=df["days"], name=f"Crude - {name}",
                             line=dict(dash=dash)), row=1, col=1)
for df, name, dash in [(prod_all_df, "All reporters", "solid"),
                       (prod_stable_df, "Stable panel", "dot")]:
    fig.add_trace(go.Scatter(x=df["date"], y=df["days"], name=f"Products - {name}",
                             line=dict(dash=dash)), row=1, col=2)
fig.update_yaxes(title_text="Days", row=1, col=1)
fig.update_yaxes(title_text="Days", row=1, col=2)
fig.update_layout(title="Days of forward demand cover", height=450, hovermode="x unified")
fig.show()

### 5.5 Demand seasonality (one line per year)

In [13]:
"""Year-over-year overlay of monthly demand. Older years faded, current year bold."""

def seasonality_chart(
    monthly: pd.DataFrame, value_col: str, value_label: str, title: str,
    start_year: int = 2015,
):
    """Generic seasonality overlay. ``monthly`` must have date + value_col."""
    df = monthly.copy()
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df["month"] = pd.to_datetime(df["date"]).dt.month
    df = df[df["year"] >= start_year]

    current_year = int(df["year"].max())
    fig = go.Figure()
    for yr, grp in df.groupby("year"):
        is_current = yr == current_year
        fig.add_trace(go.Scatter(
            x=grp["month"], y=grp[value_col],
            name=str(yr), mode="lines+markers",
            line=dict(width=3 if is_current else 1.2),
            opacity=1.0 if is_current else 0.45,
        ))
    fig.update_layout(
        title=title, height=450, hovermode="x unified",
        xaxis=dict(title="Month", tickmode="array",
                   tickvals=list(range(1, 13)),
                   ticktext=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]),
        yaxis_title=value_label,
    )
    return fig


# Demand seasonality - all reporters
seasonality_chart(
    dem_all.assign(value_mb=dem_all["obs_value"] / 1000),
    value_col="value_mb",
    value_label="mb/d",
    title="Global oil demand seasonality - all reporters (mb/d)",
).show()

### 5.6 Inventory seasonality (crude + products combined)

In [14]:
"""Crude + products inventories combined, year-over-year overlay.

Crude and product stocks are both reported in KBBL, so a simple sum is unit-correct.
"""

# Sum crude + products by date (panel = all reporters)
inv_total = (
    crude_all[["date", "obs_value"]]
    .merge(prods_all[["date", "obs_value"]], on="date", how="outer", suffixes=("_crude", "_prods"))
    .fillna(0)
)
inv_total["total_kbbl"] = inv_total["obs_value_crude"] + inv_total["obs_value_prods"]
inv_total["total_mbbl"] = inv_total["total_kbbl"] / 1000

seasonality_chart(
    inv_total[["date", "total_mbbl"]],
    value_col="total_mbbl",
    value_label="Million bbl",
    title="Global commercial stocks (crude + products) - all reporters",
).show()

### 5.7 Global trade flows (imports vs exports + closure ratio)

In a fully-reported world, every export from country A is an import to country B, so
global imports should equal global exports. The closure ratio (exports / imports) is a
proxy for reporting completeness on the trade dimension - persistent gaps between 1.0
indicate unreported barrels (often in OPEC and CIS countries).

In [15]:
"""Global imports vs exports for crude + total products combined."""

# Sum imports and exports across crude (primary) and total products (secondary)
def _trade_global(direction: str) -> pd.DataFrame:
    crude = aggregate_global(df_pri, "CRUDEOIL", direction, "KBD", panel="all")
    prods = aggregate_global(df_sec, "TOTPRODS", direction, "KBD", panel="all")
    merged = crude.merge(prods, on="date", how="outer", suffixes=("_crude", "_prods"))
    merged["obs_value"] = merged[["obs_value_crude", "obs_value_prods"]].sum(axis=1, min_count=1)
    return merged[["date", "obs_value"]]

imp = _trade_global("TOTIMPSB").rename(columns={"obs_value": "Imports"})
exp = _trade_global("TOTEXPSB").rename(columns={"obs_value": "Exports"})
trade = imp.merge(exp, on="date", how="outer").sort_values("date")
trade["Closure ratio (exp/imp)"] = trade["Exports"] / trade["Imports"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    row_heights=[0.7, 0.3],
                    subplot_titles=("Global crude+products trade (kb/d)",
                                    "Closure ratio (1.0 = balanced)"))
fig.add_trace(go.Scatter(x=trade["date"], y=trade["Imports"], name="Imports"), row=1, col=1)
fig.add_trace(go.Scatter(x=trade["date"], y=trade["Exports"], name="Exports"), row=1, col=1)
fig.add_trace(go.Scatter(x=trade["date"], y=trade["Closure ratio (exp/imp)"],
                         name="Closure ratio", line=dict(color="grey")),
              row=2, col=1)
fig.add_hline(y=1.0, line_dash="dot", line_color="red", row=2, col=1)
fig.update_layout(height=600, hovermode="x unified")
fig.update_yaxes(title_text="kb/d", row=1, col=1)
fig.update_yaxes(title_text="ratio", row=2, col=1)
fig.show()

### 5.8 Top 10 importers and exporters (latest fully-reported month)

In [16]:
"""Snapshot of the latest month: top 10 importers (left) and exporters (right) of
crude+total products combined."""

# Latest month with at least 50 reporters (proxy for "fully reported")
month_coverage = (
    df_pri.loc[(df_pri["energy_product"] == "CRUDEOIL") & (df_pri["flow_breakdown"] == "TOTIMPSB")]
          .groupby("date")["ref_area"].nunique()
)
latest_month = month_coverage[month_coverage >= 50].index.max()
print(f"Latest fully-reported month: {latest_month.date()}")


def _country_trade(direction: str) -> pd.DataFrame:
    """Crude+products import/export volumes per country, latest month."""
    crude = query_jodi(df_pri, product="CRUDEOIL", flow=direction,
                       date_min=str(latest_month.date()), date_max=str(latest_month.date()))
    prods = query_jodi(df_sec, product="TOTPRODS", flow=direction,
                       date_min=str(latest_month.date()), date_max=str(latest_month.date()))
    combined = (
        pd.concat([
            crude[["ref_area", "country_name", "obs_value"]],
            prods[["ref_area", "country_name", "obs_value"]],
        ])
        .groupby(["ref_area", "country_name"], observed=True, as_index=False)["obs_value"]
        .sum()
    )
    return combined.sort_values("obs_value", ascending=False)


imports_top = _country_trade("TOTIMPSB").head(10)
exports_top = _country_trade("TOTEXPSB").head(10)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=(f"Top 10 importers ({latest_month.date()}, kb/d)",
                                    f"Top 10 exporters ({latest_month.date()}, kb/d)"))
fig.add_trace(go.Bar(x=imports_top["country_name"], y=imports_top["obs_value"], name="Imports"),
              row=1, col=1)
fig.add_trace(go.Bar(x=exports_top["country_name"], y=exports_top["obs_value"], name="Exports"),
              row=1, col=2)
fig.update_layout(height=450, showlegend=False)
fig.update_xaxes(tickangle=-30)
fig.show()

Latest fully-reported month: 2026-03-01


### 5.9 Monthly YoY fingerprint (region x month)

Two stacked heatmaps comparing the current calendar year vs the prior year, broken down by region:

- **Top panel** - YoY % change. Diverging palette, clipped at +/- 30%.
- **Bottom panel** - Absolute change in kb/d.

Asia Pacific appears as a **consolidated** row (China + India + Other Asia Pacific).
China, India, and Other Asia Pacific also appear separately because those three move
the global aggregate so much that they deserve their own rows. The `Global Total` row is
the sum across all regions and is pinned at the top.

In [17]:
"""Region map: ISO alpha-2 -> region label.

Based on standard IEA / BP statistical groupings. Asia Pacific is available as a
consolidated row (China + India + Other Asia Pacific) and as three sub-regions so the
demand giants don't get hidden inside a regional aggregate. Anything not in this dict
falls into 'Other' and prints a warning the first time it's encountered.
"""

REGION_MAP: dict[str, str] = {
    # North America
    "US": "North America", "CA": "North America", "MX": "North America",
    # Central and South America
    "AR": "Central and South America", "BB": "Central and South America",
    "BO": "Central and South America", "BR": "Central and South America",
    "BZ": "Central and South America", "CL": "Central and South America",
    "CO": "Central and South America", "CR": "Central and South America",
    "CU": "Central and South America", "DO": "Central and South America",
    "EC": "Central and South America", "GT": "Central and South America",
    "HN": "Central and South America", "HT": "Central and South America",
    "JM": "Central and South America", "NI": "Central and South America",
    "PA": "Central and South America", "PE": "Central and South America",
    "PY": "Central and South America", "SV": "Central and South America",
    "TT": "Central and South America", "UY": "Central and South America",
    "VE": "Central and South America", "GY": "Central and South America",
    "SR": "Central and South America",
    # Europe
    "AL": "Europe", "AT": "Europe", "BA": "Europe", "BE": "Europe", "BG": "Europe",
    "CH": "Europe", "CY": "Europe", "CZ": "Europe", "DE": "Europe", "DK": "Europe",
    "EE": "Europe", "ES": "Europe", "FI": "Europe", "FR": "Europe", "GB": "Europe",
    "GR": "Europe", "HR": "Europe", "HU": "Europe", "IE": "Europe", "IS": "Europe",
    "IT": "Europe", "LT": "Europe", "LU": "Europe", "LV": "Europe", "MD": "Europe",
    "ME": "Europe", "MK": "Europe", "MT": "Europe", "NL": "Europe", "NO": "Europe",
    "PL": "Europe", "PT": "Europe", "RO": "Europe", "RS": "Europe", "SE": "Europe",
    "SI": "Europe", "SK": "Europe", "TR": "Europe", "GI": "Europe", "FO": "Europe",
    # CIS (former Soviet states)
    "AM": "CIS", "AZ": "CIS", "BY": "CIS", "GE": "CIS", "KG": "CIS", "KZ": "CIS",
    "RU": "CIS", "TJ": "CIS", "TM": "CIS", "UA": "CIS", "UZ": "CIS",
    # Middle East
    "AE": "Middle East", "BH": "Middle East", "IL": "Middle East", "IQ": "Middle East",
    "IR": "Middle East", "JO": "Middle East", "KW": "Middle East", "LB": "Middle East",
    "OM": "Middle East", "PS": "Middle East", "QA": "Middle East", "SA": "Middle East",
    "SY": "Middle East", "YE": "Middle East",
    # Africa
    "AO": "Africa", "BJ": "Africa", "BW": "Africa", "CD": "Africa", "CG": "Africa",
    "CI": "Africa", "CM": "Africa", "DJ": "Africa", "DZ": "Africa", "EG": "Africa",
    "ER": "Africa", "ET": "Africa", "GA": "Africa", "GH": "Africa", "GM": "Africa",
    "GN": "Africa", "GQ": "Africa", "KE": "Africa", "LR": "Africa", "LS": "Africa",
    "LY": "Africa", "MA": "Africa", "MG": "Africa", "ML": "Africa", "MR": "Africa",
    "MU": "Africa", "MW": "Africa", "MZ": "Africa", "NA": "Africa", "NE": "Africa",
    "NG": "Africa", "RW": "Africa", "SC": "Africa", "SD": "Africa", "SL": "Africa",
    "SN": "Africa", "SO": "Africa", "SS": "Africa", "ST": "Africa", "SZ": "Africa",
    "TD": "Africa", "TG": "Africa", "TN": "Africa", "TZ": "Africa", "UG": "Africa",
    "ZA": "Africa", "ZM": "Africa", "ZW": "Africa", "BF": "Africa", "BI": "Africa",
    "CF": "Africa", "KM": "Africa", "CV": "Africa",
    # Asia Pacific - consolidated row built in yoy_fingerprint(); China/India broken out
    "CN": "China",
    "IN": "India",
    # Other Asia Pacific
    "AU": "Other Asia Pacific", "BD": "Other Asia Pacific", "BN": "Other Asia Pacific",
    "BT": "Other Asia Pacific", "FJ": "Other Asia Pacific", "HK": "Other Asia Pacific",
    "ID": "Other Asia Pacific", "JP": "Other Asia Pacific", "KH": "Other Asia Pacific",
    "KR": "Other Asia Pacific", "KP": "Other Asia Pacific", "LA": "Other Asia Pacific",
    "LK": "Other Asia Pacific", "MM": "Other Asia Pacific", "MN": "Other Asia Pacific",
    "MO": "Other Asia Pacific", "MV": "Other Asia Pacific", "MY": "Other Asia Pacific",
    "NP": "Other Asia Pacific", "NZ": "Other Asia Pacific", "PG": "Other Asia Pacific",
    "PH": "Other Asia Pacific", "PK": "Other Asia Pacific", "SG": "Other Asia Pacific",
    "TH": "Other Asia Pacific", "TL": "Other Asia Pacific", "TW": "Other Asia Pacific",
    "VN": "Other Asia Pacific", "AF": "Other Asia Pacific",
    # Bermuda is in N. America geographically, but JODI lists it separately
    "BM": "North America",
}

# Sub-regions that roll up into consolidated Asia Pacific (heatmap + dashboard).
_ASIA_PACIFIC_PARTS = frozenset({"China", "India", "Other Asia Pacific"})
CONSOLIDATED_ASIA_PACIFIC = "Asia Pacific"

# Display order for the heatmap rows (Global Total goes on top, prepended at render time)
REGION_ORDER = [
    CONSOLIDATED_ASIA_PACIFIC,
    "China", "India", "Other Asia Pacific",
    "CIS", "Middle East", "Africa",
    "Europe", "Central and South America", "North America",
]

# Diagnostic: any reporters in the data not covered by the map?
_known_codes = set(REGION_MAP)
_data_codes = set(df_sec["ref_area"].astype(str).unique())
_unmapped = sorted(_data_codes - _known_codes)
if _unmapped:
    print(f"WARNING: {len(_unmapped)} ref_area codes not in REGION_MAP, will fall into 'Other':")
    print(f"  {_unmapped}")
else:
    print(f"All {len(_data_codes)} ref_area codes mapped to a region.")

  ['GD']


In [18]:
"""Build the YoY fingerprint and render the two-panel heatmap.

Wrapped in a function so the same chart can be regenerated for any (df, product, flow,
unit) tuple by changing one argument - e.g. inventories, exports, crude production.
"""

def yoy_fingerprint(
    df: pd.DataFrame, product: str, flow: str, unit: str = "KBD",
    current_year: Optional[int] = None,
    title_suffix: str = "",
) -> tuple[pd.DataFrame, pd.DataFrame, go.Figure]:
    """Return (yoy_pct_pivot, delta_pivot, plotly_figure) for region x month."""
    if current_year is None:
        current_year = int(df["year"].max())
    prior_year = current_year - 1

    # Slice both years and tag with region.
    sub = query_jodi(
        df, product=product, flow=flow, unit=unit,
        date_min=f"{prior_year}-01-01", date_max=f"{current_year}-12-01",
    )
    sub["region"] = sub["ref_area"].astype(str).map(REGION_MAP).fillna("Other")

    # Aggregate to region x month x year (mean across countries to keep volume sensible
    # for kb/d - using sum here because we want regional totals in absolute volume terms).
    grouped = (
        sub.groupby(["region", "year", "month"], observed=True, as_index=False)["obs_value"]
           .sum()
    )

    # Consolidated Asia Pacific = China + India + Other Asia Pacific (sum of sub-regions).
    asia_pacific = (
        grouped[grouped["region"].isin(_ASIA_PACIFIC_PARTS)]
        .groupby(["year", "month"], as_index=False)["obs_value"]
        .sum()
        .assign(region=CONSOLIDATED_ASIA_PACIFIC)
    )
    grouped = pd.concat([grouped, asia_pacific], ignore_index=True)

    # Global Total from country-level data (avoids double-counting APAC sub-regions
    # once the consolidated Asia Pacific row is appended above).
    global_total = (
        sub.groupby(["year", "month"], as_index=False)["obs_value"]
        .sum()
        .assign(region="Global Total")
    )
    grouped = pd.concat([grouped, global_total], ignore_index=True)

    # Pivot each year to a (region x month) frame
    cur = grouped[grouped["year"] == current_year].pivot(index="region", columns="month", values="obs_value")
    pri = grouped[grouped["year"] == prior_year].pivot(index="region", columns="month", values="obs_value")

    all_months = list(range(1, 13))
    cur = cur.reindex(columns=all_months)
    pri = pri.reindex(columns=all_months)

    # YoY % and delta
    yoy_pct = ((cur - pri) / pri) * 100.0
    delta = cur - pri

    # Row order: Global Total on top, then the regions we actually have data for
    # (intersected with REGION_ORDER for stable ordering, plus any 'Other' at the end).
    present = set(yoy_pct.index)
    ordered_rows = ["Global Total"] + [r for r in REGION_ORDER if r in present]
    if "Other" in present:
        ordered_rows.append("Other")
    yoy_pct = yoy_pct.reindex(ordered_rows)
    delta = delta.reindex(ordered_rows)

    month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    title = (f"Monthly YoY fingerprint: {product} x {flow} x {unit} "
             f"(<b>{current_year} vs {prior_year}</b>){title_suffix}")

    # Coerce nullable Float64 -> plain numpy float64 (np.nan for missing) so
    # downstream numpy ops (np.isnan, vectorize) behave normally.
    yoy_arr   = yoy_pct.to_numpy(dtype="float64", na_value=np.nan)
    delta_arr = delta.to_numpy(dtype="float64", na_value=np.nan)

    def _fmt(arr: np.ndarray, fmt: str) -> np.ndarray:
        out = np.empty(arr.shape, dtype=object)
        mask = ~np.isnan(arr)
        out[mask] = [format(v, fmt) for v in arr[mask]]
        out[~mask] = ""
        return out

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=("YoY % change", f"Delta {unit}"),
    )

    fig.add_trace(go.Heatmap(
        z=yoy_arr,
        x=month_labels, y=yoy_pct.index.tolist(),
        colorscale="RdYlGn", zmin=-30, zmax=30, zmid=0,
        colorbar=dict(title="YoY %", len=0.45, y=0.78, x=1.02),
        text=_fmt(yoy_arr, "+.1f"),
        texttemplate="%{text}", textfont=dict(size=10),
        hovertemplate="%{y} | %{x}: %{z:+.1f}%<extra></extra>",
    ), row=1, col=1)

    # Auto-clip the absolute scale so small regions stay visible
    finite = delta_arr[np.isfinite(delta_arr)]
    delta_max = max(abs(finite).max() if finite.size else 1.0, 1.0)
    delta_clip = max(100.0, min(delta_max, 1500.0))

    fig.add_trace(go.Heatmap(
        z=delta_arr,
        x=month_labels, y=delta.index.tolist(),
        colorscale="RdYlGn", zmin=-delta_clip, zmax=delta_clip, zmid=0,
        colorbar=dict(title=f"\u0394 {unit}", len=0.45, y=0.22, x=1.02),
        text=_fmt(delta_arr, "+.0f"),
        texttemplate="%{text}", textfont=dict(size=10),
        hovertemplate="%{y} | %{x}: %{z:+.0f} " + unit + "<extra></extra>",
    ), row=2, col=1)

    fig.update_layout(title=title, height=700)
    fig.update_yaxes(autorange="reversed")  # Global Total on top
    return yoy_pct, delta, fig


# Default chart: total products demand (kb/d), current year vs prior
yoy_pct_dem, delta_dem, fig_dem = yoy_fingerprint(
    df_sec, product="TOTPRODS", flow="TOTDEMO", unit="KBD",
)
fig_dem.show()

In [19]:
"""Variants of the same fingerprint for inventories, exports, imports, and crude
production. Comment out any you don't need."""

# Crude production (primary)
_, _, fig_crude_prod = yoy_fingerprint(
    df_pri, product="CRUDEOIL", flow="INDPROD", unit="KBD",
    title_suffix=" - crude production",
)
fig_crude_prod.show()

# Closing stocks (products) - in KBBL since stocks are levels not flows
_, _, fig_inv = yoy_fingerprint(
    df_sec, product="TOTPRODS", flow="CLOSTLV", unit="KBBL",
    title_suffix=" - product inventories",
)
fig_inv.show()

# Imports (crude+products combined would be ideal, but this gives the products view)
_, _, fig_imp = yoy_fingerprint(
    df_sec, product="TOTPRODS", flow="TOTIMPSB", unit="KBD",
    title_suffix=" - product imports",
)
fig_imp.show()

# Exports
_, _, fig_exp = yoy_fingerprint(
    df_sec, product="TOTPRODS", flow="TOTEXPSB", unit="KBD",
    title_suffix=" - product exports",
)
fig_exp.show()

In [24]:
"""Country-count diagnostic - how many reporters fed each (region, month) cell?

Use this to sanity-check whether a swing in the heatmap above is driven by a missing
reporter rather than real demand change. A cell can look red just because Iraq stopped
reporting that month.
"""

current_year = int(df_sec["year"].max())
prior_year   = current_year - 1
sub = query_jodi(
    df_sec, product="TOTPRODS", flow="TOTDEMO", unit="KBD",
    date_min=f"{prior_year}-01-01", date_max=f"{current_year}-12-01",
)
sub["region"] = sub["ref_area"].astype(str).map(REGION_MAP).fillna("Other")

coverage = (
    sub.groupby(["year", "region", "month"], observed=True)["ref_area"]
       .nunique()
       .unstack("month")
       .fillna(0)
       .astype(int)
)

# Consolidated Asia Pacific row: unique reporters across CN, IN, and Other APAC.
ap_coverage = (
    sub[sub["region"].isin(_ASIA_PACIFIC_PARTS)]
    .groupby(["year", "month"], observed=True)["ref_area"]
    .nunique()
    .reset_index()
    .assign(region=CONSOLIDATED_ASIA_PACIFIC)
)
ap_wide = (
    ap_coverage
    .pivot(index=["year", "region"], columns="month", values="ref_area")
    .reindex(columns=coverage.columns)
    .fillna(0)  # pivot + reindex can leave NA; must fill before int cast
    .astype(int)
)
# Drop a stray column if an earlier buggy run created "Asia Pacific" as a column.
if CONSOLIDATED_ASIA_PACIFIC in coverage.columns:
    coverage = coverage.drop(columns=[CONSOLIDATED_ASIA_PACIFIC])
coverage = pd.concat([coverage, ap_wide])
coverage = coverage[~coverage.index.duplicated(keep="last")]

print("Reporter counts per (region, month) - current year:")
coverage.loc[current_year]

Reporter counts per (region, month) - current year:


month,1,2,3,4,5,6,7,8,9,10,11,12
region,,,,,,,,,,,,
Africa,2,2,2,0,0,0,0,0,0,0,0,0
CIS,2,1,0,0,0,0,0,0,0,0,0,0
Central and South America,2,2,2,0,0,0,0,0,0,0,0,0
China,1,1,1,0,0,0,0,0,0,0,0,0
Europe,34,34,27,0,0,0,0,0,0,0,0,0
India,1,1,1,0,0,0,0,0,0,0,0,0
Middle East,3,3,2,0,0,0,0,0,0,0,0,0
North America,3,3,2,0,0,0,0,0,0,0,0,0
Other Asia Pacific,10,10,10,0,0,0,0,0,0,0,0,0


## 6. SQL access example

Same data is mirrored to SQLite (`jodi_secondary.db`, `jodi_primary.db`) with indexes on
`(date, ref_area)`, `energy_product`, and `flow_breakdown`. Use this when you prefer a
quick SQL idiom over pandas filtering.

> Note: Parquet is more efficient for analytical queries (compressed columnar with
> categorical dictionaries). SQLite is here for ad-hoc filters and for non-Python tools
> (DBeaver, tableau, etc.).

In [21]:
"""Top 10 product demanders for the latest full year, via SQLite."""

if SQLITE_SECONDARY.exists():
    with sqlite3.connect(SQLITE_SECONDARY) as con:
        sql_top = pd.read_sql(f"""
            SELECT ref_area, country_name,
                   ROUND(AVG(obs_value), 1) AS mean_kbd,
                   COUNT(*) AS n_months
            FROM jodi_secondary
            WHERE date LIKE '{latest_full_year}-%'
              AND energy_product = 'TOTPRODS'
              AND flow_breakdown = 'TOTDEMO'
              AND unit_measure   = 'KBD'
              AND value_status   = 'valid'
            GROUP BY ref_area, country_name
            HAVING COUNT(*) >= 10
            ORDER BY mean_kbd DESC
            LIMIT 10
        """, con)
    print(f"Top 10 oil consumers ({latest_full_year}, mean kb/d, via SQLite)")
    display(sql_top)
else:
    print(f"SQLite mirror not found at {SQLITE_SECONDARY}")
    print("Run: python scripts/update_jodi.py --bootstrap   (without --no-sqlite)")

SQLite mirror not found at c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed\jodi\jodi_secondary.db
Run: python scripts/update_jodi.py --bootstrap   (without --no-sqlite)


## 7. Snowflake hand-off

The Parquet files are written with snappy compression and a flat schema, ready for direct
ingestion via Snowflake's `COPY INTO` from an external stage.

### Suggested DDL

```sql
CREATE OR REPLACE TABLE jodi_secondary (
    date              DATE,
    year              SMALLINT,
    month             TINYINT,
    ref_area          VARCHAR(8),
    country_name      VARCHAR(128),
    energy_product    VARCHAR(16),
    flow_breakdown    VARCHAR(16),
    unit_measure      VARCHAR(16),
    obs_value         NUMBER(18, 4),
    value_status      VARCHAR(16),
    assessment_code   TINYINT,
    assessment_label  VARCHAR(16),
    dataset           VARCHAR(16),
    source_file       VARCHAR(64),
    updated_at        TIMESTAMP_NTZ
)
CLUSTER BY (year, ref_area);   -- prunes by date & country, the most common filter
```

`jodi_primary` follows the identical schema (only `dataset` and the product/flow values
differ).

### Loading from the local Parquet

```sql
-- 1. Create or use an existing internal stage
CREATE STAGE IF NOT EXISTS jodi_stage
    FILE_FORMAT = (TYPE = PARQUET);

-- 2. Upload the parquet (Snowflake CLI / SnowSQL):
--    PUT file:///<path>/jodi_secondary.parquet @jodi_stage AUTO_COMPRESS=FALSE;

-- 3. COPY INTO with column matching by name (Parquet stores them so we don't need a list)
COPY INTO jodi_secondary
FROM @jodi_stage/jodi_secondary.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
```

### Suggested partitioning / clustering

For ~15 M rows the table is small by Snowflake standards, but if you union with other
data sources later, cluster by `(YEAR, REF_AREA)` so country time-series filters prune
micro-partitions. Avoid clustering on `DATE` directly because the cardinality (~290
distinct months) is too low to be useful.

### Refresh pattern

The local pipeline rebuilds the parquet on each `update_jodi.py` run. To mirror this in
Snowflake without truncating downstream tables, use a `MERGE INTO` with the natural key
`(date, ref_area, energy_product, flow_breakdown, unit_measure)` and source it from the
freshly loaded parquet stage.